# 数据路径

In [ ]:
work_dir = "original-layout/Projects/Jupyter/deepseek-prover-V2/runs/FormalMATH_eval_results"
sum_path_0 = work_dir + "/s0-20251204-144105/summary.jsonl"
sum_path_1 = work_dir + "/s1-20251204-144207/summary.jsonl"
sum_path_2 = work_dir + "/s2-20260103-235415/summary.jsonl"

In [ ]:

minif2f_sum_path = "original-layout/Projects/Jupyter/deepseek-prover-V2/runs/MiniF2F_eval/results/summary.jsonl"

In [ ]:
s = work_dir + sum_path_0
print(s)

# 综合分析

In [ ]:
import json, jsonlines
from pathlib import Path

import json, jsonlines, math
from pathlib import Path
import time
import re



def summarize_from_summary(summary_file: str):
    """
    给定一个 summary.jsonl 文件，统计：
      - problem_pass_rate
      - answer_pass_rate
      - first_pass_rate
      - first_pass_over_passed
    统计：
      - all_time:  min / max / avg
      - pass_time: min / max / avg（仅 pass==True 的样本）
      - successes: min / max / avg（每题通过的样本数）

    并打印出来，同时返回一个 dict。
    """
    summary_path = Path(summary_file)
    if not summary_path.exists():
        print(f"[summarize] {summary_file} 不存在，无法统计")
        return {}

    total_problems = 0
    passed_problems = 0
    total_answers = 0
    passed_answers = 0
    first_pass_problems = 0
    
    # ---- time_sec（全部样本）----
    all_cnt = 0
    all_sum = 0.0
    all_min = None
    all_max = None
    
    # ---- time_sec（仅通过样本）----
    pass_cnt = 0
    pass_sum = 0.0
    pass_min = None
    pass_max = None

    # ---- successes（按 problem 统计）----
    succ_cnt = 0
    succ_sum = 0.0
    succ_min = None
    succ_max = None
    
    # 记录总样本数的观察值：避免最后只打印到最后一行的 n_s
    last_n_s = None
    
    
    
    def _accum(t: float, is_pass: bool):
        nonlocal all_cnt, all_sum, all_min, all_max
        nonlocal pass_cnt, pass_sum, pass_min, pass_max

        # all
        all_cnt += 1
        all_sum += t
        all_min = t if all_min is None else min(all_min, t)
        all_max = t if all_max is None else max(all_max, t)

        # pass-only
        if is_pass:
            pass_cnt += 1
            pass_sum += t
            pass_min = t if pass_min is None else min(pass_min, t)
            pass_max = t if pass_max is None else max(pass_max, t)
    
    

    with open(summary_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue

            total_problems += 1

            if row.get("pass", False):
                passed_problems += 1
                
            if row.get("first_pass", False):
                first_pass_problems += 1

            # 每题生成多少个回答
            n_s = row.get("n_samples")
            if n_s is None:
                # 兜底：没有 n_samples 就用 attempts 的长度
                n_s = len(row.get("attempts", []))
            try:
                n_s = int(n_s)
            except Exception:
                n_s = 0
            last_n_s = n_s

            successes = row.get("successes", 0)
            try:
                successes = int(successes)
            except Exception:
                successes = 0

            # ---- successes 统计（按 problem 一行一个）----
            succ_cnt += 1
            succ_sum += float(successes)
            succ_min = successes if succ_min is None else min(succ_min, successes)
            succ_max = successes if succ_max is None else max(succ_max, successes)

            total_answers += n_s
            passed_answers += successes
            

            # -------- time_sec：优先从 attempts[*].time_sec 读 --------
            attempts = row.get("attempts")
            if isinstance(attempts, list) and attempts:
                for att in attempts:
                    t = att.get("time_sec", None)
                    if t is None:
                        continue
                    try:
                        t = float(t)
                    except Exception:
                        continue
                    _accum(t, is_pass=(att.get("pass") is True))
            else:
                # 旧结构兜底：row.time_sec
                t = row.get("time_sec", None)
                if t is not None:
                    try:
                        t = float(t)
                        _accum(t, is_pass=row_pass)
                    except Exception:
                        pass
    
    
    
    print("\n[SUMMARY FROM FILE]")
    print(f"  samples (per problem): {n_s}")
    
    # 1) 题通过率：有题过就算过
    problem_pass_rate = 100.0 * passed_problems / total_problems if total_problems else 0.0
    print(f"  problem_pass_rate: total = {total_problems}  passed = {passed_problems}  (problem_pass_rate={problem_pass_rate:.2f}%)")

    # 2) 回答通过率：所有通过的回答 / 所有生成的回答
    answer_pass_rate = 100.0 * passed_answers / total_answers if total_answers else 0.0
    print(f"  answer_pass_rate : total = {total_answers}  passed = {passed_answers}  (answer_pass_rate={answer_pass_rate:.2f}%)")
    
    # 3) 第一通过率：第一个回答就通过的题目 / 所有题目
    first_pass_rate   = 100.0 * first_pass_problems / total_problems if total_problems else 0.0
    print(f"  first_pass_rate: first_pass = {first_pass_problems}  (first_pass_rate = {first_pass_rate:.2f}%)")
    
    # 4) 第一通过占比：第一个回答就通过的题目 / 总通过的题目数
    first_pass_over_passed = (
        100.0 * first_pass_problems / passed_problems if passed_problems else 0.0
    )
    print(f"  first_pass_over_passed: {first_pass_over_passed:.2f}%")        
    
    
    
    # ---- time 输出 ----  succ_count:一共查看了几条数据
    all_avg = (all_sum / all_cnt) if all_cnt else None
    pass_avg = (pass_sum / pass_cnt) if pass_cnt else None

    if all_cnt:
        print(f"  time_sec (ALL):  succ_count = {all_cnt}  min = {all_min:.4f}  max = {all_max:.4f}  avg = {all_avg:.4f}")
    else:
        print("  time_sec (ALL):  没有找到可用数据")

    if pass_cnt:
        print(f"  time_sec (PASS): succ_count = {pass_cnt}  min = {pass_min:.4f}  max = {pass_max:.4f}  avg = {pass_avg:.4f}")
    else:
        print("  time_sec (PASS): 没有找到可用数据（或没有 pass==True 的样本）")

        
        
    # ---- successes 输出（按 problem）----  succ_count:一共查看了几条数据
    succ_avg = (succ_sum / succ_cnt) if succ_cnt else None
    if succ_cnt:
        print(f"  successes (PER PROBLEM): succ_count = {succ_cnt}  min = {succ_min}  max = {succ_max}  avg = {succ_avg:.4f}")
    else:
        print("  successes (PER PROBLEM): 没有找到可用数据")

    
    

    return dict(
        total_problems = total_problems,
        passed_problems = passed_problems,
        first_pass_problems = first_pass_problems,
        total_answers = total_answers,
        passed_answers = passed_answers,
        problem_pass_rate = problem_pass_rate,
        answer_pass_rate = answer_pass_rate,
        first_pass_rate = first_pass_rate,
        first_pass_over_passed = first_pass_over_passed,
        all_time_count=all_cnt,
        all_time_min=all_min,
        all_time_max=all_max,
        all_time_avg=all_avg,
        pass_time_count=pass_cnt,
        pass_time_min=pass_min,
        pass_time_max=pass_max,
        pass_time_avg=pass_avg,

        # ---- successes 统计返回 ----
        successes_count=succ_cnt,
        successes_min=succ_min,
        successes_max=succ_max,
        successes_avg=succ_avg,
    )


In [ ]:
sum_minif2f = summarize_from_summary(minif2f_sum_path)

In [ ]:

sum0 = summarize_from_summary(sum_path_0)

In [ ]:
sum1 = summarize_from_summary(sum_path_1)

In [ ]:
sum2 = summarize_from_summary(sum_path_2)

# 时间分析

In [ ]:
out_path_0 = work_dir + "/s0-20251204-144105/time_analyze"
out_path_1 = work_dir + "/s1-20251204-144207/time_analyze"

In [ ]:
import json, math, hashlib
from pathlib import Path
import jsonlines
import numpy as np
import matplotlib.pyplot as plt


def _normalize_newlines_keep_ws(s: str) -> str:
    # 只统一换行符，不做 strip，避免丢缩进/空白
    return s.replace("\r\n", "\n").replace("\r", "\n")


def _resolve_file(summary_path: Path, p: str) -> Path:
    pp = Path(p)
    if pp.exists():
        return pp
    return summary_path.parent / pp


def _get_lean_block_keep_ws(summary_path: Path, row: dict, att: dict | None) -> tuple[str | None, str | None]:
    """
    返回 (lean_block, lean_path)：
    - 优先 attempt["lean_block"]
    - 否则尝试从 attempt["lean_path"] 读取
    """
    lean_block = None
    lean_path = None

    if att is not None:
        lb = att.get("lean_block")
        if isinstance(lb, str) and lb != "":
            lean_block = _normalize_newlines_keep_ws(lb)

        lp = att.get("lean_path")
        if isinstance(lp, str) and lp != "":
            lean_path = lp

    # 回退从文件读（只有 lean_block 为空时才读）
    if (lean_block is None or lean_block == "") and lean_path:
        p = _resolve_file(summary_path, lean_path)
        if p.exists():
            lean_block = _normalize_newlines_keep_ws(p.read_text(encoding="utf-8"))

    return lean_block, lean_path


def _iter_attempt_records(summary_path: Path, row: dict):
    """
    兼容两种结构：
    - 新结构：row["attempts"] 是 list
    - 旧结构：row 自己有 pass/time_sec/lean_block/lean_path
    逐条 yield record(dict)

    ✅增强：把 reason/log_path/header 也写进 record
    """
    rid = str(row.get("id") or "")
    name = row.get("name")
    split = row.get("split")
    header = row.get("header") or ""   # 原样保留

    attempts = row.get("attempts")
    if isinstance(attempts, list) and attempts:
        for att in attempts:
            idx = int(att.get("index", 1))
            new_id = f"{rid}_s{idx:02d}" if rid else f"s{idx:02d}"
            passed = (att.get("pass") is True)

            time_sec = att.get("time_sec", None)
            try:
                time_sec = float(time_sec) if time_sec is not None else None
            except Exception:
                time_sec = None

            lean_block, lean_path = _get_lean_block_keep_ws(summary_path, row, att)

            # ✅新增：reason / log_path（reason 有时在 row 上也可能出现，attempt 优先）
            reason = att.get("reason", None)
            if reason is None:
                reason = row.get("reason", None)

            log_path = att.get("log_path", None)
            if log_path is None:
                log_path = row.get("log_path", None)

            yield {
                "id": new_id,
                "source_id": rid,
                "sample_index": idx,
                "name": name,
                "split": split,
                "header": header,
                "pass": passed,
                "time_sec": time_sec,
                "lean_block": lean_block,
                "lean_path": lean_path,
                "reason": reason,       # ✅新增
                "log_path": log_path,   # ✅保留/强化
            }
    else:
        # 旧结构：把 row 当作一个样本
        new_id = f"{rid}_s01" if rid else "s01"
        passed = (row.get("pass") is True)

        time_sec = row.get("time_sec", None)
        try:
            time_sec = float(time_sec) if time_sec is not None else None
        except Exception:
            time_sec = None

        # 旧结构 lean_block / lean_path 在 row 上
        lb = row.get("lean_block")
        lean_block = _normalize_newlines_keep_ws(lb) if isinstance(lb, str) and lb != "" else None
        lean_path = row.get("lean_path", None)

        if (lean_block is None or lean_block == "") and isinstance(lean_path, str) and lean_path != "":
            p = _resolve_file(summary_path, lean_path)
            if p.exists():
                lean_block = _normalize_newlines_keep_ws(p.read_text(encoding="utf-8"))

        yield {
            "id": new_id,
            "source_id": rid,
            "sample_index": 1,
            "name": name,
            "split": split,
            "header": header,
            "pass": passed,
            "time_sec": time_sec,
            "lean_block": lean_block,
            "lean_path": lean_path,
            "reason": row.get("reason", None),     # ✅新增
            "log_path": row.get("log_path", None), # ✅新增/保留
        }


def _save_jsonl(path: Path, records: list[dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with jsonlines.open(path, "w") as w:
        for r in records:
            w.write(r)


def _hist_counts(times: list[float], target_bins: int = 60):
    """
    生成直方图数据：bin_edges, counts
    bins 选择为“约 target_bins 个箱”，保证横轴是时间、纵轴是数量
    """
    if not times:
        return [], []
    tmin, tmax = min(times), max(times)
    if tmax <= tmin:
        edges = [tmin, tmin + 1e-9]
        counts = [len(times)]
        return edges, counts

    width = max(1e-6, (tmax - tmin) / float(target_bins))
    nbins = max(1, int(math.ceil((tmax - tmin) / width)))
    edges = [tmin + i * width for i in range(nbins + 1)]
    counts, _ = np.histogram(np.array(times, dtype=float), bins=np.array(edges, dtype=float))
    return edges, counts.tolist()


def _plot_hist(edges, counts, out_png: Path, title: str):
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.figure()
    if edges and counts:
        left = np.array(edges[:-1], dtype=float)
        width = np.array(edges[1:], dtype=float) - left
        plt.bar(left, counts, width=width, align="edge")
        plt.xlabel("time_sec")
        plt.ylabel("count")
        plt.title(title)
    else:
        plt.title(title + " (no valid time_sec)")
        plt.xlabel("time_sec")
        plt.ylabel("count")
    plt.tight_layout()
    plt.savefig(out_png)
    plt.show()


def _compute_stats(records: list[dict], thresholds_sec: list[int]):
    times = [r["time_sec"] for r in records if isinstance(r.get("time_sec"), (int, float))]
    missing = sum(1 for r in records if not isinstance(r.get("time_sec"), (int, float)))

    stat = {
        "n_records": len(records),
        "n_with_time": len(times),
        "n_missing_time": missing,
        "avg_time_sec": (sum(times) / len(times)) if times else None,
        "min_time_sec": (min(times) if times else None),
        "max_time_sec": (max(times) if times else None),
        "min_record": None,
        "max_record": None,
        "thresholds": {},
        "hist": {"bin_edges": [], "counts": []},
    }

    if times:
        tmin = stat["min_time_sec"]
        tmax = stat["max_time_sec"]

        # ✅增强：min/max 对应记录里也保留 header/reason/log_path
        def _pick_by_time(target):
            for r in records:
                if r.get("time_sec") == target:
                    return {
                        "id": r.get("id"),
                        "time_sec": r.get("time_sec"),
                        "header": r.get("header"),
                        "reason": r.get("reason"),
                        "log_path": r.get("log_path"),
                        "lean_path": r.get("lean_path"),
                        "lean_block": r.get("lean_block"),
                    }
            return None

        stat["min_record"] = _pick_by_time(tmin)
        stat["max_record"] = _pick_by_time(tmax)

        edges, counts = _hist_counts(times, target_bins=60)
        stat["hist"]["bin_edges"] = edges
        stat["hist"]["counts"] = counts

    # 阈值统计：> 30/60/90min
    for th in thresholds_sec:
        over = [
            {
                "id": r.get("id"),
                "time_sec": r.get("time_sec"),
                "header": r.get("header"),
                "reason": r.get("reason"),
                "log_path": r.get("log_path"),
                "lean_path": r.get("lean_path"),
                "lean_block": r.get("lean_block"),
            }
            for r in records
            if isinstance(r.get("time_sec"), (int, float)) and r["time_sec"] > th
        ]
        stat["thresholds"][f"> {th} sec"] = {
            "threshold_sec": th,
            "count": len(over),
            "items": over,
        }

    return stat


def time_analysis_from_summary(
    summary_file: str,
    out_dir: str = "time_analysis",
    thresholds_min: list[int] = [30, 60, 90],
):
    """
    输入：summary.jsonl
    输出：
      - out_dir/records_all.jsonl
      - out_dir/records_pass.jsonl
      - out_dir/records_fail.jsonl
      - out_dir/records_over_60min.jsonl    ✅新增
      - out_dir/time_stats.json
      - out_dir/hist_all.png / hist_pass.png / hist_fail.png

    同时“直接输出”：三类的 avg/min/max 以及对应 id，并展示三张直方图。
    """
    summary_path = Path(summary_file)
    if not summary_path.exists():
        raise FileNotFoundError(f"{summary_file} 不存在")

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    thresholds_sec = [int(m) * 60 for m in thresholds_min]

    all_records = []
    pass_records = []
    fail_records = []

    with jsonlines.open(summary_path, "r") as reader:
        for row in reader:
            for rec in _iter_attempt_records(summary_path, row):
                all_records.append(rec)
                if rec.get("pass") is True:
                    pass_records.append(rec)
                else:
                    fail_records.append(rec)

    # 分别写三份 records.jsonl
    _save_jsonl(out_dir / "records_all.jsonl", all_records)
    _save_jsonl(out_dir / "records_pass.jsonl", pass_records)
    _save_jsonl(out_dir / "records_fail.jsonl", fail_records)

    # ✅新增：> 60min 的 records 单独存一份 jsonl（保留完整字段）
    over_60min_sec = 60 * 60
    over_60min_records = [
        r for r in all_records
        if isinstance(r.get("time_sec"), (int, float)) and r["time_sec"] > over_60min_sec
    ]
    _save_jsonl(out_dir / "records_over_60min.jsonl", over_60min_records)

    # 计算统计并写 time_stats.json
    stats_all = _compute_stats(all_records, thresholds_sec)
    stats_pass = _compute_stats(pass_records, thresholds_sec)
    stats_fail = _compute_stats(fail_records, thresholds_sec)

    stats = {
        "summary_file": str(summary_path),
        "thresholds_min": thresholds_min,
        "ALL": stats_all,
        "PASS": stats_pass,
        "FAIL": stats_fail,
    }
    (out_dir / "time_stats.json").write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")

    # 直接输出：三类 avg/min/max + id
    def _print_scope(scope_name: str, st: dict):
        print(f"\n[{scope_name}]")
        print(f"  n_records={st['n_records']}  n_with_time={st['n_with_time']}  n_missing_time={st['n_missing_time']}")
        print(f"  avg_time_sec={st['avg_time_sec']}")
        if st["min_record"]:
            print(f"  min_time_sec={st['min_time_sec']}  id={st['min_record']['id']}")
        else:
            print(f"  min_time_sec=None")
        if st["max_record"]:
            print(f"  max_time_sec={st['max_time_sec']}  id={st['max_record']['id']}")
        else:
            print(f"  max_time_sec=None")

    _print_scope("ALL", stats_all)
    _print_scope("PASS", stats_pass)
    _print_scope("FAIL", stats_fail)

    # 展示直方图（横轴时间，纵轴数量），并保存 png
    _plot_hist(stats_all["hist"]["bin_edges"], stats_all["hist"]["counts"], out_dir / "hist_all.png", "ALL time_sec histogram")
    _plot_hist(stats_pass["hist"]["bin_edges"], stats_pass["hist"]["counts"], out_dir / "hist_pass.png", "PASS time_sec histogram")
    _plot_hist(stats_fail["hist"]["bin_edges"], stats_fail["hist"]["counts"], out_dir / "hist_fail.png", "FAIL time_sec histogram")

    print(f"\n[OK] wrote:")
    print(f"  - {out_dir/'records_all.jsonl'}")
    print(f"  - {out_dir/'records_pass.jsonl'}")
    print(f"  - {out_dir/'records_fail.jsonl'}")
    print(f"  - {out_dir/'records_over_60min.jsonl'}")  # ✅新增
    print(f"  - {out_dir/'time_stats.json'}")
    print(f"  - hist_*.png in {out_dir}")

    return stats


In [ ]:
import json, math, hashlib
from pathlib import Path
import jsonlines
import numpy as np
import matplotlib.pyplot as plt


def _normalize_newlines_keep_ws(s: str) -> str:
    return s.replace("\r\n", "\n").replace("\r", "\n")


def _resolve_file(summary_path: Path, p: str) -> Path:
    pp = Path(p)
    if pp.exists():
        return pp
    return summary_path.parent / pp


def _get_lean_block_keep_ws(summary_path: Path, row: dict, att: dict | None) -> tuple[str | None, str | None]:
    lean_block = None
    lean_path = None

    if att is not None:
        lb = att.get("lean_block")
        if isinstance(lb, str) and lb != "":
            lean_block = _normalize_newlines_keep_ws(lb)

        lp = att.get("lean_path")
        if isinstance(lp, str) and lp != "":
            lean_path = lp

    if (lean_block is None or lean_block == "") and lean_path:
        p = _resolve_file(summary_path, lean_path)
        if p.exists():
            lean_block = _normalize_newlines_keep_ws(p.read_text(encoding="utf-8"))

    return lean_block, lean_path


def _iter_attempt_records(summary_path: Path, row: dict):
    rid = str(row.get("id") or "")
    name = row.get("name")
    split = row.get("split")
    header = row.get("header") or ""

    attempts = row.get("attempts")
    if isinstance(attempts, list) and attempts:
        for att in attempts:
            idx = int(att.get("index", 1))
            new_id = f"{rid}_s{idx:02d}" if rid else f"s{idx:02d}"
            passed = (att.get("pass") is True)

            time_sec = att.get("time_sec", None)
            try:
                time_sec = float(time_sec) if time_sec is not None else None
            except Exception:
                time_sec = None

            lean_block, lean_path = _get_lean_block_keep_ws(summary_path, row, att)

            reason = att.get("reason", None)
            if reason is None:
                reason = row.get("reason", None)

            log_path = att.get("log_path", None)
            if log_path is None:
                log_path = row.get("log_path", None)

            # ✅新增：domain（attempt 优先，其次 row；都没有则 None）
            domain = att.get("domain", None)
            if domain is None:
                domain = row.get("domain", None)

            yield {
                "id": new_id,
                "source_id": rid,
                "sample_index": idx,
                "name": name,
                "split": split,
                "header": header,
                "pass": passed,
                "time_sec": time_sec,
                "lean_block": lean_block,
                "lean_path": lean_path,
                "reason": reason,
                "log_path": log_path,
                "domain": domain,        # ✅新增
            }
    else:
        new_id = f"{rid}_s01" if rid else "s01"
        passed = (row.get("pass") is True)

        time_sec = row.get("time_sec", None)
        try:
            time_sec = float(time_sec) if time_sec is not None else None
        except Exception:
            time_sec = None

        lb = row.get("lean_block")
        lean_block = _normalize_newlines_keep_ws(lb) if isinstance(lb, str) and lb != "" else None
        lean_path = row.get("lean_path", None)

        if (lean_block is None or lean_block == "") and isinstance(lean_path, str) and lean_path != "":
            p = _resolve_file(summary_path, lean_path)
            if p.exists():
                lean_block = _normalize_newlines_keep_ws(p.read_text(encoding="utf-8"))

        yield {
            "id": new_id,
            "source_id": rid,
            "sample_index": 1,
            "name": name,
            "split": split,
            "header": header,
            "pass": passed,
            "time_sec": time_sec,
            "lean_block": lean_block,
            "lean_path": lean_path,
            "reason": row.get("reason", None),
            "log_path": row.get("log_path", None),
            "domain": row.get("domain", None),   # ✅新增
        }


def _save_jsonl(path: Path, records: list[dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with jsonlines.open(path, "w") as w:
        for r in records:
            w.write(r)


def _hist_counts(times: list[float], target_bins: int = 60):
    if not times:
        return [], []
    tmin, tmax = min(times), max(times)
    if tmax <= tmin:
        edges = [tmin, tmin + 1e-9]
        counts = [len(times)]
        return edges, counts

    width = max(1e-6, (tmax - tmin) / float(target_bins))
    nbins = max(1, int(math.ceil((tmax - tmin) / width)))
    edges = [tmin + i * width for i in range(nbins + 1)]
    counts, _ = np.histogram(np.array(times, dtype=float), bins=np.array(edges, dtype=float))
    return edges, counts.tolist()


def _plot_hist(edges, counts, out_png: Path, title: str):
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.figure()
    if edges and counts:
        left = np.array(edges[:-1], dtype=float)
        width = np.array(edges[1:], dtype=float) - left
        plt.bar(left, counts, width=width, align="edge")
        plt.xlabel("time_sec")
        plt.ylabel("count")
        plt.title(title)
    else:
        plt.title(title + " (no valid time_sec)")
        plt.xlabel("time_sec")
        plt.ylabel("count")
    plt.tight_layout()
    plt.savefig(out_png)
    plt.show()


def _compute_stats(records: list[dict], thresholds_sec: list[int]):
    times = [r["time_sec"] for r in records if isinstance(r.get("time_sec"), (int, float))]
    missing = sum(1 for r in records if not isinstance(r.get("time_sec"), (int, float)))

    stat = {
        "n_records": len(records),
        "n_with_time": len(times),
        "n_missing_time": missing,
        "avg_time_sec": (sum(times) / len(times)) if times else None,
        "min_time_sec": (min(times) if times else None),
        "max_time_sec": (max(times) if times else None),
        "min_record": None,
        "max_record": None,
        "thresholds": {},
        "hist": {"bin_edges": [], "counts": []},
    }

    if times:
        tmin = stat["min_time_sec"]
        tmax = stat["max_time_sec"]

        def _pick_by_time(target):
            for r in records:
                if r.get("time_sec") == target:
                    return {
                        "id": r.get("id"),
                        "time_sec": r.get("time_sec"),
                        "domain": r.get("domain"),     # ✅新增
                        "header": r.get("header"),
                        "reason": r.get("reason"),
                        "log_path": r.get("log_path"),
                        "lean_path": r.get("lean_path"),
                        "lean_block": r.get("lean_block"),
                    }
            return None

        stat["min_record"] = _pick_by_time(tmin)
        stat["max_record"] = _pick_by_time(tmax)

        edges, counts = _hist_counts(times, target_bins=60)
        stat["hist"]["bin_edges"] = edges
        stat["hist"]["counts"] = counts

    for th in thresholds_sec:
        over = [
            {
                "id": r.get("id"),
                "time_sec": r.get("time_sec"),
                "domain": r.get("domain"),   # ✅新增
                "header": r.get("header"),
                "reason": r.get("reason"),
                "log_path": r.get("log_path"),
                "lean_path": r.get("lean_path"),
                "lean_block": r.get("lean_block"),
            }
            for r in records
            if isinstance(r.get("time_sec"), (int, float)) and r["time_sec"] > th
        ]
        stat["thresholds"][f"> {th} sec"] = {
            "threshold_sec": th,
            "count": len(over),
            "items": over,
        }

    return stat


def _domain_share(records: list[dict]) -> dict:
    """
    统计 records 中 domain 占比（domain 允许为 None）
    返回 dict：total / counts / shares / sorted
    注意：JSON object 的 key 必须是 str，所以 None 会写成 "null"
    """
    total = len(records)
    counts = {}
    for r in records:
        d = r.get("domain", None)
        key = d if isinstance(d, str) and d != "" else "null"
        counts[key] = counts.get(key, 0) + 1

    shares = {k: (v / total if total else 0.0) for k, v in counts.items()}
    sorted_items = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)

    return {
        "total": total,
        "counts": counts,
        "shares": shares,
        "sorted": [{"domain": k, "count": v, "share": (v / total if total else 0.0)} for k, v in sorted_items],
    }


def time_analysis_from_summary(
    summary_file: str,
    out_dir: str = "time_analysis",
    thresholds_min: list[int] = [30, 60, 90],
):
    summary_path = Path(summary_file)
    if not summary_path.exists():
        raise FileNotFoundError(f"{summary_file} 不存在")

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    thresholds_sec = [int(m) * 60 for m in thresholds_min]

    all_records = []
    pass_records = []
    fail_records = []

    with jsonlines.open(summary_path, "r") as reader:
        for row in reader:
            for rec in _iter_attempt_records(summary_path, row):
                all_records.append(rec)
                if rec.get("pass") is True:
                    pass_records.append(rec)
                else:
                    fail_records.append(rec)

    _save_jsonl(out_dir / "records_all.jsonl", all_records)
    _save_jsonl(out_dir / "records_pass.jsonl", pass_records)
    _save_jsonl(out_dir / "records_fail.jsonl", fail_records)

    over_60min_sec = 60 * 60
    over_60min_records = [
        r for r in all_records
        if isinstance(r.get("time_sec"), (int, float)) and r["time_sec"] > over_60min_sec
    ]
    _save_jsonl(out_dir / "records_over_60min.jsonl", over_60min_records)

    # ✅新增：>60min 的 domain 占比分析（保存 + 简洁输出）
    domain_stats_over_60min = _domain_share(over_60min_records)
    (out_dir / "domain_over_60min.json").write_text(
        json.dumps(domain_stats_over_60min, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    print("\n[>60min domain share]")
    for item in domain_stats_over_60min["sorted"][:20]:  # 只打印前20个，避免刷屏
        print(f"  domain={item['domain']}  count={item['count']}  share={item['share']:.4f}")

    stats_all = _compute_stats(all_records, thresholds_sec)
    stats_pass = _compute_stats(pass_records, thresholds_sec)
    stats_fail = _compute_stats(fail_records, thresholds_sec)

    stats = {
        "summary_file": str(summary_path),
        "thresholds_min": thresholds_min,
        "ALL": stats_all,
        "PASS": stats_pass,
        "FAIL": stats_fail,
        # ✅可选：把 >60min 的 domain 分析也塞进主 stats（不影响原逻辑）
        "OVER_60MIN_DOMAIN": domain_stats_over_60min,
    }
    (out_dir / "time_stats.json").write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")

    def _print_scope(scope_name: str, st: dict):
        print(f"\n[{scope_name}]")
        print(f"  n_records={st['n_records']}  n_with_time={st['n_with_time']}  n_missing_time={st['n_missing_time']}")
        print(f"  avg_time_sec={st['avg_time_sec']}")
        if st["min_record"]:
            print(f"  min_time_sec={st['min_time_sec']}  id={st['min_record']['id']}")
        else:
            print(f"  min_time_sec=None")
        if st["max_record"]:
            print(f"  max_time_sec={st['max_time_sec']}  id={st['max_record']['id']}")
        else:
            print(f"  max_time_sec=None")

    _print_scope("ALL", stats_all)
    _print_scope("PASS", stats_pass)
    _print_scope("FAIL", stats_fail)

    _plot_hist(stats_all["hist"]["bin_edges"], stats_all["hist"]["counts"], out_dir / "hist_all.png", "ALL time_sec histogram")
    _plot_hist(stats_pass["hist"]["bin_edges"], stats_pass["hist"]["counts"], out_dir / "hist_pass.png", "PASS time_sec histogram")
    _plot_hist(stats_fail["hist"]["bin_edges"], stats_fail["hist"]["counts"], out_dir / "hist_fail.png", "FAIL time_sec histogram")

    print(f"\n[OK] wrote:")
    print(f"  - {out_dir/'records_all.jsonl'}")
    print(f"  - {out_dir/'records_pass.jsonl'}")
    print(f"  - {out_dir/'records_fail.jsonl'}")
    print(f"  - {out_dir/'records_over_60min.jsonl'}")
    print(f"  - {out_dir/'domain_over_60min.json'}")   # ✅新增
    print(f"  - {out_dir/'time_stats.json'}")
    print(f"  - hist_*.png in {out_dir}")

    return stats


In [ ]:
stats_0 = time_analysis_from_summary(
    sum_path_0,
    out_path_0,
    thresholds_min=[30, 60, 90],
)

stats_1 = time_analysis_from_summary(
    sum_path_1,
    out_path_1,
    thresholds_min=[30, 60, 90],
)

# 1 

In [ ]:
import json
import math
import statistics
import re
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple, Union
from collections import Counter, defaultdict

# -----------------------------
# I/O: load records
# -----------------------------
def _load_records(inp: Union[str, Path, List[dict], Iterable[dict]]) -> List[dict]:
    if isinstance(inp, (list, tuple)):
        return [dict(x) for x in inp]
    if not isinstance(inp, (str, Path)):
        return [dict(x) for x in inp]

    s = str(inp)
    p = Path(s)
    if p.exists() and p.is_file():
        recs = []
        with open(p, "r", encoding="utf-8") as f:
            for ln in f:
                ln = ln.strip()
                if not ln:
                    continue
                recs.append(json.loads(ln))
        return recs

    recs = []
    for ln in s.splitlines():
        ln = ln.strip()
        if not ln:
            continue
        recs.append(json.loads(ln))
    return recs


# -----------------------------
# helpers: error extraction & categorization
# -----------------------------
_FIRST_ERROR_RE = re.compile(r"error:\s*(.+)", re.IGNORECASE)

def _get_error_text(rec: dict) -> str:
    return (rec.get("stdout_tail") or "") + "\n" + (rec.get("stderr_tail") or "")

def _extract_first_error_line(err_text: str) -> str:
    for ln in (err_text or "").splitlines():
        if "error:" in ln:
            return ln.strip()
    return ""

def _has_warning(rec: dict) -> bool:
    t = (rec.get("stdout_tail") or "") + "\n" + (rec.get("stderr_tail") or "")
    return ("warning:" in t)

def _classify_failure(tag: str, err_text: str, returncode: Any) -> str:
    tag = (tag or "").strip()
    t = (err_text or "").lower()

    known_fine = {
        "syntax_error", "unsolved_goals", "type_mismatch", "typeclass_failure",
        "timeout", "lake_not_found", "no_goals", "linarith_failed", "simp_failed",
        "rewrite_failed", "interval_cases", "tactic_failed", "other_rc1","unknown_identifier",
    }
    if tag in known_fine:
        return tag

    if tag in {"timeout", "lake_not_found"}:
        return tag

    if not t.strip() and returncode not in (0, None):
        return f"rc_{returncode}"

    if "unsolved goals" in t:
        return "unsolved_goals"
    
    if "unexpected token" in t or "expected ')'" in t or "expected ']'" in t or "expected term" in t:
        return "syntax_error"
    
    if "parser error" in t:
        return "syntax_error"
    
    if "type mismatch" in t:
        return "type_mismatch"
    
    # unknown identifier / constant（名字解析失败）
    if "unknown identifier" in t or "unknown constant" in t:
        return "unknown_identifier"

    
    if "failed to synthesize" in t or "failed to find instance" in t or "typeclass" in t:
        return "typeclass_failure"
    
    if "interval_cases failed" in t:
        return "interval_cases"
    
    if "linarith failed" in t:
        return "linarith_failed"
    
    if "tactic 'simp' failed" in t or "simp' failed" in t:
        return "simp_failed"
    
    if "tactic 'rewrite' failed" in t or "did not find instance of the pattern" in t:
        return "rewrite_failed"
    
    if "no goals to be solved" in t:
        return "no_goals"
    
    if "tactic failed" in t:
        return "tactic_failed"

    return tag if tag else "other"

def _percent(x: int, n: int) -> float:
    return 0.0 if n == 0 else (100.0 * x / n)

def _quantile(xs: List[float], q: float) -> Optional[float]:
    if not xs:
        return None
    xs = sorted(xs)
    if q <= 0:
        return xs[0]
    if q >= 1:
        return xs[-1]
    pos = (len(xs) - 1) * q
    lo = int(math.floor(pos))
    hi = int(math.ceil(pos))
    if lo == hi:
        return xs[lo]
    return xs[lo] * (hi - pos) + xs[hi] * (pos - lo)

def _time_pack(xs: List[float]) -> Dict[str, Optional[float]]:
    if not xs:
        return {"mean": None, "median": None, "p90": None, "p95": None, "max": None}
    return {
        "mean": sum(xs) / len(xs),
        "median": statistics.median(xs),
        "p90": _quantile(xs, 0.90),
        "p95": _quantile(xs, 0.95),
        "max": max(xs),
    }

# -----------------------------
# NEW helpers for pass@k setting
# -----------------------------
_GENUID_RE = re.compile(r"__a(\d+)$")

def _infer_case_id(rr: dict, idx: int) -> str:
    """
    题目级唯一 key（用于聚合多 attempt）。
    优先 id；否则 name/source_id/sample_index；再否则 idx（不推荐，但保底）。
    """
    rid = rr.get("id")
    if isinstance(rid, str) and rid.strip():
        return rid.strip()

    name = rr.get("name")
    source_id = rr.get("source_id")
    sample_index = rr.get("sample_index")
    parts = []
    if name not in (None, "", "null"):
        parts.append(f"name={name}")
    if source_id not in (None, "", "null"):
        parts.append(f"source_id={source_id}")
    if sample_index not in (None, "", "null"):
        parts.append(f"sample_index={sample_index}")

    if parts:
        return " | ".join(parts)

    return f"IDX={idx}"

def _infer_attempt_idx(rr: dict) -> int:
    """
    attempt index：
    1) gen_attempt
    2) 从 gen_uid / gen_case_key / lean_path 里解析 __aXX
    3) 默认 0
    """
    ga = rr.get("gen_attempt")
    if isinstance(ga, int):
        return ga
    if isinstance(ga, str) and ga.strip().isdigit():
        return int(ga.strip())

    for k in ["gen_uid", "gen_case_key", "lean_path"]:
        v = rr.get(k)
        if isinstance(v, str) and v:
            m = _GENUID_RE.search(v)
            if m:
                try:
                    return int(m.group(1))
                except Exception:
                    pass
    return 0

def _infer_gen_uid(case_id: str, attempt_idx: int) -> str:
    return f"{case_id}__a{attempt_idx:02d}"

def _sorted_attempts(attempt_recs: List[dict]) -> List[dict]:
    return sorted(attempt_recs, key=lambda x: (x["_attempt_idx"], str(x.get("gen_uid") or "")))



def _get_sketch_text(rr: dict, sketch_key_candidates: Tuple[str, ...] = ("sketch", "sketch_origin", "sketch_raw")) -> str:
    for k in sketch_key_candidates:
        v = rr.get(k)
        if isinstance(v, str) and v.strip():
            return v
    return ""

def _infer_hole_count_for_case(attempts: List[dict], proof_token: str = "<proof>") -> Optional[int]:
    """
    从一个 case 的 attempts 中推断“原 sketch 的洞数”。
    规则：找到第一条能拿到 sketch 的 attempt，用其 sketch.count("<proof>")。
    """
    for a in attempts:
        s = _get_sketch_text(a)
        if s:
            return s.count(proof_token)
    return None

def _hole_bucket_name(h: Optional[int]) -> str:
    if h is None:
        return "unknown_holes"
    if h == 1:
        return "single_hole"
    if h > 1:
        return "multi_hole"
    return "unknown_holes"



# -----------------------------
# main: analysis (multi-attempt aware)
# -----------------------------
def summarize_verify_records_multi(
    inp: Union[str, Path, List[dict], Iterable[dict]],
    bench_key_candidates: Tuple[str, ...] = ("source_bench", "source_bech"),
    top_k: int = 20,
    show_examples_per_reason: int = 3,
    pass_at_k_list: Tuple[int, ...] = (1, 2, 3, 5, 10),
    max_k_from_data: bool = True,
    return_summary: bool = True,
) -> Dict[str, Any]:
    """
    适配 verify 结果“同一题目多 attempt（pass@k）”的统计。

    输出两套视角：
      - Attempt-level: 每一行 verify 记录当一个样本
      - Case-level: 按题目聚合，计算 pass@k

    failure 也分：
      - attempt_failure_*: 对所有失败尝试做分类统计
      - case_failure_*: 对“在 max_k 下仍失败的题目”，用代表尝试（默认 attempt 最小）做分类统计
    """

    recs = _load_records(inp)
    total_lines = len(recs)

    # -------- normalize records (attempt-level) --------
    norm: List[dict] = []
    for idx, r in enumerate(recs):
        rr = dict(r)

        # bench
        bench = "UNKNOWN"
        for k in bench_key_candidates:
            if rr.get(k) not in (None, "", "null"):
                bench = rr.get(k)
                break

        # ids
        case_id = _infer_case_id(rr, idx)
        attempt_idx = _infer_attempt_idx(rr)

        # ensure gen_uid exists (do not overwrite if user already wrote it)
        if not (isinstance(rr.get("gen_uid"), str) and rr.get("gen_uid").strip()):
            rr["gen_uid"] = _infer_gen_uid(case_id, attempt_idx)

        passed = bool(rr.get("pass", False))
        tag = rr.get("tag", "") or ""
        rc = rr.get("returncode", None)
        time_sec = rr.get("time_sec", None)
        name = rr.get("name", "") or ""

        err_text = _get_error_text(rr)
        first_error = _extract_first_error_line(err_text)
        has_warn = _has_warning(rr)

        if passed:
            failure_cat = "pass"
        else:
            failure_cat = _classify_failure(tag, err_text, rc)

        rr["_bench"] = bench
        rr["_case_id"] = case_id
        rr["_attempt_idx"] = attempt_idx
        rr["_passed"] = passed
        rr["_tag"] = tag
        rr["_rc"] = rc
        rr["_time"] = time_sec if isinstance(time_sec, (int, float)) else None
        rr["_name"] = name
        rr["_failure_cat"] = failure_cat
        rr["_first_error"] = first_error
        rr["_has_warning"] = has_warn

        norm.append(rr)

    # group by bench -> case_id -> attempts
    bench_case_map: Dict[str, Dict[str, List[dict]]] = defaultdict(lambda: defaultdict(list))
    for rr in norm:
        bench_case_map[rr["_bench"]][rr["_case_id"]].append(rr)

    # overall case map
    overall_case_map: Dict[str, List[dict]] = defaultdict(list)
    for rr in norm:
        overall_case_map[rr["_case_id"]].append(rr)

    # determine max_k
    max_attempts_overall = 0
    for cid, attempts in overall_case_map.items():
        max_attempts_overall = max(max_attempts_overall, len(_sorted_attempts(attempts)))

    k_list = list(pass_at_k_list)
    if max_k_from_data and max_attempts_overall > 0:
        k_list.append(max_attempts_overall)
    k_list = sorted({k for k in k_list if isinstance(k, int) and k >= 1})
    if max_attempts_overall > 0:
        k_list = [k for k in k_list if k <= max_attempts_overall] or [max_attempts_overall]
    max_k = max(k_list) if k_list else 1

    # -------- summarizers --------
    def _summ_attempt_level(group_attempts: List[dict]) -> Dict[str, Any]:
        n = len(group_attempts)
        passed_n = sum(1 for x in group_attempts if x["_passed"])
        failed_n = n - passed_n
        warn_n = sum(1 for x in group_attempts if x["_has_warning"])

        fail_cats = [x["_failure_cat"] for x in group_attempts if not x["_passed"]]
        fail_counter = Counter(fail_cats)

        tag_counter = Counter([x["_tag"] or "" for x in group_attempts if not x["_passed"]])
        rc_counter = Counter([str(x["_rc"]) for x in group_attempts if not x["_passed"]])

        first_errs = [x["_first_error"] for x in group_attempts if (not x["_passed"] and x["_first_error"])]
        first_err_counter = Counter(first_errs)

        fail_names = [x["_name"] for x in group_attempts if (not x["_passed"] and x["_name"])]
        fail_name_counter = Counter(fail_names)

        times_all = [x["_time"] for x in group_attempts if isinstance(x["_time"], (int, float))]
        times_pass = [x["_time"] for x in group_attempts if x["_passed"] and isinstance(x["_time"], (int, float))]
        times_fail = [x["_time"] for x in group_attempts if (not x["_passed"]) and isinstance(x["_time"], (int, float))]

        fail_ratios = {k: _percent(v, failed_n) for k, v in fail_counter.most_common()}

        return {
            "attempt_n": n,
            "attempt_passed": passed_n,
            "attempt_failed": failed_n,
            "attempt_pass_rate": _percent(passed_n, n),
            "attempt_warning_count": warn_n,
            "attempt_warning_rate": _percent(warn_n, n),
            "attempt_failure_category_counts": dict(fail_counter),
            "attempt_failure_category_ratios_over_failed(%)": fail_ratios,
            "attempt_raw_tag_counts": dict(tag_counter),
            "attempt_returncode_counts": dict(rc_counter),
            "attempt_top_first_error": first_err_counter.most_common(top_k),
            "attempt_top_failed_names": fail_name_counter.most_common(top_k),
            "attempt_time_all": _time_pack(times_all),
            "attempt_time_pass": _time_pack(times_pass),
            "attempt_time_fail": _time_pack(times_fail),
        }

    def _summ_case_level(case_map: Dict[str, List[dict]]) -> Dict[str, Any]:
        """
        case_map: case_id -> list of attempts
        """
        case_n = len(case_map)
        
        
        # NEW: hole-bucket pass@k
        hole_bucket_counts = Counter()  # bucket -> number of cases
        pass_at_k_by_hole: Dict[str, Dict[int, int]] = defaultdict(lambda: defaultdict(int))  # bucket -> k -> solved_count

        

        # pass@k
        pass_at_k: Dict[int, Dict[str, Any]] = {}
        solved_cases_at_max_k: List[str] = []
        unsolved_cases_at_max_k: List[str] = []

        # for optional stats
        first_pass_attempt_pos: List[int] = []   # 1-based position in sorted attempts (for cases that solved)
        first_pass_time: List[float] = []        # time_sec of first passing attempt (if exists)

        # representative record for unsolved cases at max_k (used in failure breakdown)
        rep_unsolved_records: List[dict] = []

        for cid, attempts in case_map.items():
            ats = _sorted_attempts(attempts)
            
            
            # NEW: determine hole bucket for this case
            holes = _infer_hole_count_for_case(ats, proof_token="<proof>")
            hb = _hole_bucket_name(holes)
            hole_bucket_counts[hb] += 1


            # compute solved position/time
            solved_pos = None
            solved_time = None
            for j, a in enumerate(ats, start=1):
                if a["_passed"]:
                    solved_pos = j
                    solved_time = a["_time"] if isinstance(a["_time"], (int, float)) else None
                    break
            if solved_pos is not None:
                first_pass_attempt_pos.append(solved_pos)
                if solved_time is not None:
                    first_pass_time.append(solved_time)

            # pass@k decisions
            for k in k_list:
                topk = ats[:k]
                ok = any(a["_passed"] for a in topk)
                pass_at_k.setdefault(k, {"solved": 0})
                if ok:
                    pass_at_k[k]["solved"] += 1

                # NEW: hole-bucket pass@k
                if ok:
                    pass_at_k_by_hole[hb][k] += 1


            # max_k solved/unsolved sets
            maxk_ok = any(a["_passed"] for a in ats[:max_k])
            if maxk_ok:
                solved_cases_at_max_k.append(cid)
            else:
                unsolved_cases_at_max_k.append(cid)
                # representative attempt: smallest attempt_idx among first max_k
                rep = ats[0] if ats else None
                if rep is not None:
                    rep_unsolved_records.append(rep)

        # finalize pass@k rates
        pass_at_k_rates = {}
        for k in k_list:
            solved = pass_at_k.get(k, {}).get("solved", 0)
            pass_at_k_rates[k] = {
                "solved_cases": solved,
                "total_cases": case_n,
                "pass_at_k(%)": _percent(solved, case_n),
            }
            
            
            
        # NEW: finalize hole-bucket pass@k rates
        pass_at_k_by_hole_rates: Dict[str, Dict[int, Dict[str, Any]]] = {}
        for hb, total_hb in hole_bucket_counts.items():
            per_k = {}
            for k in k_list:
                solved = int(pass_at_k_by_hole[hb].get(k, 0))
                per_k[k] = {
                    "solved_cases": solved,
                    "total_cases": int(total_hb),
                    "pass_at_k(%)": _percent(solved, int(total_hb)),
                }
            pass_at_k_by_hole_rates[hb] = per_k


        # failure breakdown on unsolved cases at max_k
        fail_cats = [x["_failure_cat"] for x in rep_unsolved_records]  # all are failed
        fail_counter = Counter(fail_cats)
        failed_n = len(rep_unsolved_records)
        fail_ratios = {k: _percent(v, failed_n) for k, v in fail_counter.most_common()}

        # first_error top (unsolved case reps)
        first_errs = [x["_first_error"] for x in rep_unsolved_records if x["_first_error"]]
        first_err_counter = Counter(first_errs)

        # failed names top (unsolved case reps)
        fail_names = [x["_name"] for x in rep_unsolved_records if x["_name"]]
        fail_name_counter = Counter(fail_names)

        return {
            "case_n": case_n,
            "k_list": k_list,
            "max_k": max_k,
            "pass_at_k": pass_at_k_rates,
            
            "pass_at_k_by_hole": pass_at_k_by_hole_rates,
            "hole_bucket_case_counts": dict(hole_bucket_counts),

            "case_solved_at_max_k": len(solved_cases_at_max_k),
            "case_unsolved_at_max_k": len(unsolved_cases_at_max_k),
            "case_pass_at_max_k(%)": _percent(len(solved_cases_at_max_k), case_n),

            # how fast it solves (optional)
            "case_first_pass_attempt_pos_stats": {
                "mean": (sum(first_pass_attempt_pos) / len(first_pass_attempt_pos)) if first_pass_attempt_pos else None,
                "median": statistics.median(first_pass_attempt_pos) if first_pass_attempt_pos else None,
                "p90": _quantile([float(x) for x in first_pass_attempt_pos], 0.90) if first_pass_attempt_pos else None,
                "max": max(first_pass_attempt_pos) if first_pass_attempt_pos else None,
                "n_solved": len(first_pass_attempt_pos),
            },
            "case_first_pass_time_stats": _time_pack(first_pass_time),

            # unsolved reason breakdown (representative attempt per case)
            "case_failure_category_counts_unsolved_at_max_k": dict(fail_counter),
            "case_failure_category_ratios_over_unsolved(%)": fail_ratios,
            "case_top_first_error_unsolved_at_max_k": first_err_counter.most_common(top_k),
            "case_top_failed_names_unsolved_at_max_k": fail_name_counter.most_common(top_k),

            # store reps for examples
            "_rep_unsolved_records": rep_unsolved_records,
        }

    def _fmt_time(d: Dict[str, Optional[float]]) -> str:
        def f(x):
            return "NA" if x is None else f"{x:.3f}"
        return f"mean={f(d['mean'])}  median={f(d['median'])}  p90={f(d['p90'])}  p95={f(d['p95'])}  max={f(d['max'])}"

    def _print_report(title: str, attempts: List[dict], case_map: Dict[str, List[dict]]):
        attempt_s = _summ_attempt_level(attempts)
        case_s = _summ_case_level(case_map)

        print("\n" + "=" * 90)
        print(f"[{title}]")
        print(f"Attempts: Total={attempt_s['attempt_n']} | PASS={attempt_s['attempt_passed']} ({attempt_s['attempt_pass_rate']:.2f}%) | FAIL={attempt_s['attempt_failed']} ({100.0 - attempt_s['attempt_pass_rate']:.2f}%)")
        print(f"Cases   : Total={case_s['case_n']} | Solved@max_k={case_s['case_solved_at_max_k']} ({case_s['case_pass_at_max_k(%)']:.2f}%) | Unsolved@max_k={case_s['case_unsolved_at_max_k']}")
        print(f"Warnings(attempt-level)={attempt_s['attempt_warning_count']} ({attempt_s['attempt_warning_rate']:.2f}%)")
        print("-" * 90)

        # pass@k table
        print("pass@k (case-level):")
        for k in case_s["k_list"]:
            r = case_s["pass_at_k"][k]
            print(f"  - pass@{k:<2d}: {r['solved_cases']:5d}/{r['total_cases']:<5d} ({r['pass_at_k(%)']:.2f}%)")

            
            
        # NEW: pass@k by hole bucket
        if "pass_at_k_by_hole" in case_s:
            print("\npass@k by hole-bucket (case-level):")
            counts = case_s.get("hole_bucket_case_counts", {})
            for hb in ["single_hole", "multi_hole", "unknown_holes"]:
                if hb not in case_s["pass_at_k_by_hole"]:
                    continue
                total_hb = counts.get(hb, 0)
                print(f"  [{hb}] total_cases={total_hb}")
                for k in case_s["k_list"]:
                    r = case_s["pass_at_k_by_hole"][hb][k]
                    print(f"    - pass@{k:<2d}: {r['solved_cases']:5d}/{r['total_cases']:<5d} ({r['pass_at_k(%)']:.2f}%)")

            
            
        # attempt-level failure categories
        if attempt_s["attempt_failed"] > 0:
            print("\nAttempt-level failure categories (count / % over failed attempts):")
            for k, c in sorted(attempt_s["attempt_failure_category_counts"].items(), key=lambda x: x[1], reverse=True):
                pct = attempt_s["attempt_failure_category_ratios_over_failed(%)"].get(k, 0.0)
                print(f"  - {k:18s} : {c:5d}  ({pct:6.2f}%)")

        # case-level unsolved failure categories at max_k
        if case_s["case_unsolved_at_max_k"] > 0:
            print(f"\nCase-level failure categories (UNSOLVED at max_k={case_s['max_k']}, representative attempt) (count / % over unsolved cases):")
            for k, c in sorted(case_s["case_failure_category_counts_unsolved_at_max_k"].items(), key=lambda x: x[1], reverse=True):
                pct = case_s["case_failure_category_ratios_over_unsolved(%)"].get(k, 0.0)
                print(f"  - {k:18s} : {c:5d}  ({pct:6.2f}%)")

            # examples per reason (unsolved reps)
            if show_examples_per_reason > 0:
                print("\nExamples per failure category (UNSOLVED reps):")
                by_cat = defaultdict(list)
                for x in case_s["_rep_unsolved_records"]:
                    by_cat[x["_failure_cat"]].append(x)

                for cat, _ in sorted(case_s["case_failure_category_counts_unsolved_at_max_k"].items(), key=lambda x: x[1], reverse=True):
                    xs = by_cat.get(cat, [])
                    if not xs:
                        continue
                    print(f"  * {cat}:")
                    for ex in xs[:show_examples_per_reason]:
                        print(f"      id={ex.get('id')}  gen_uid={ex.get('gen_uid')}  attempt={ex.get('gen_attempt')}  name={ex.get('name')}  rc={ex.get('returncode')}")
                        fe = ex.get("_first_error") or ""
                        if fe:
                            print(f"        first_error: {fe[:200]}")
                        else:
                            et = _get_error_text(ex).strip().splitlines()
                            if et:
                                print(f"        err_line: {et[0][:200]}")

        # time stats
        print("\nTime stats (sec) (attempt-level):")
        print("  all : " + _fmt_time(attempt_s["attempt_time_all"]))
        print("  pass: " + _fmt_time(attempt_s["attempt_time_pass"]))
        print("  fail: " + _fmt_time(attempt_s["attempt_time_fail"]))

        print("\nFirst-pass attempt position stats (case-level, solved cases only):")
        ap = case_s["case_first_pass_attempt_pos_stats"]
        if ap["n_solved"] > 0:
            print(f"  n_solved={ap['n_solved']} | mean={ap['mean']:.3f} | median={ap['median']:.3f} | p90={ap['p90']:.3f} | max={ap['max']}")
        else:
            print("  n_solved=0")

        # top first_error (unsolved reps)
        if case_s["case_top_first_error_unsolved_at_max_k"]:
            print(f"\nTop first_error (UNSOLVED reps, max_k={case_s['max_k']}):")
            for msg, c in case_s["case_top_first_error_unsolved_at_max_k"][:min(top_k, 10)]:
                print(f"  - ({c}) {msg[:200]}")

        # top failed names (unsolved reps)
        if case_s["case_top_failed_names_unsolved_at_max_k"]:
            print(f"\nTop failed theorem names (UNSOLVED reps, max_k={case_s['max_k']}):")
            for nm, c in case_s["case_top_failed_names_unsolved_at_max_k"][:min(top_k, 10)]:
                print(f"  - ({c}) {nm}")

    # -------- overall report --------
    overall_attempts = norm
    overall_case_map2 = overall_case_map

    overall_attempt_s = _summ_attempt_level(overall_attempts)
    overall_case_s = _summ_case_level(overall_case_map2)
    _print_report("OVERALL", overall_attempts, overall_case_map2)

    # -------- by bench report --------
    by_bench = {}
    for bench, case_map in sorted(bench_case_map.items(), key=lambda kv: len(kv[1]), reverse=True):
        attempts = []
        for cid, ats in case_map.items():
            attempts.extend(ats)

        attempt_s = _summ_attempt_level(attempts)
        case_s = _summ_case_level(case_map)
        _print_report(f"BENCH={bench}", attempts, case_map)

        # 去掉内部字段再放 summary
        case_s2 = dict(case_s)
        case_s2.pop("_rep_unsolved_records", None)

        by_bench[bench] = {
            "attempt_level": attempt_s,
            "case_level": case_s2,
        }

    overall_case_s2 = dict(overall_case_s)
    overall_case_s2.pop("_rep_unsolved_records", None)

    summary = {
        "meta": {
            "total_lines(attempts)": total_lines,
            "total_cases": len(overall_case_map2),
            "k_list": k_list,
            "max_k": max_k,
        },
        "overall": {
            "attempt_level": overall_attempt_s,
            "case_level": overall_case_s2,
        },
        "by_bench": by_bench,
    }
    return summary if return_summary else {}


In [ ]:
VERIFY_JSONL = "original-layout/Projects/Jupyter/SketchFill/holewise_training_2/test_runs/Sequential_filling_test/Sequential_filling_test_results/verify_results.jsonl"

In [ ]:
summary = summarize_verify_records_multi(
    VERIFY_JSONL,
    pass_at_k_list=(1, 12, 24, 32),
    show_examples_per_reason=3,
)

In [ ]:
p = "original-layout/Projects/Jupyter/SketchFill/holewise_training_2/test_runs/parallel_filling_test/parallel_filling_test_results/verify_results.jsonl"

In [ ]:
summary = summarize_verify_records_multi(
    p,
    pass_at_k_list=(1, 12, 24, 32),
    show_examples_per_reason=3,
)